# Lab: preparing the Titanic for a model

Lab of the lesson [Data Preprocessing](https://insper.github.io/ML/2026.2/classes/preprocessing/): from the raw Titanic CSV to a numeric matrix with no holes, no text and comparable scales, every statistic learned on the training rows only. The notebook follows the two labs of the page: the **fundamental** lab (steps 1 to 6) first, then the **advanced** one (steps 7 to 9), in the same session. Each step opens with a **Predict**: answer it before running the cell. The solutions of the simulations are on the page.

## Fundamental lab — preparing the Titanic, column by column

### Step 1 — load and take inventory

**Predict.** Before looking: which columns of a passenger list do you expect to have holes? Which ones are text, and which numbers are not really quantities?

In [ ]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula04/titanic/train.csv"
df = pd.read_csv(url)
print(df.shape)
inventory = pd.DataFrame({'numeric': [pd.api.types.is_numeric_dtype(df[c]) for c in df.columns],
                          'missing': df.isna().sum(),
                          'distinct': df.nunique()})
print(inventory)

### Step 2 — split before anything else

**Predict.** The split comes before any imputation or scaling. What would go wrong if you computed the median age on all 891 passengers and only then split?

In [ ]:
from sklearn.model_selection import train_test_split

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X, y = df[features], df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print("missing in train:", X_train.isna().sum()[lambda s: s > 0].to_dict())
print("missing in test: ", X_test.isna().sum()[lambda s: s > 0].to_dict())

### Step 3 — why is Age missing?

**Predict.** Do you expect age to be missing equally often in the three classes? And would deleting the passengers without an age change what the training set looks like?

In [ ]:
missing = X_train['Age'].isna()
print("share of missing Age by class:", missing.groupby(X_train['Pclass']).mean().round(3).to_dict())
print("survival rate, Age present / missing:", y_train.groupby(missing).mean().round(3).to_dict())
kept = X_train.dropna(subset=['Age'])
print(f"dropna keeps {len(kept)} of {len(X_train)} rows; "
      f"3rd-class share {(X_train['Pclass'] == 3).mean():.3f} -> {(kept['Pclass'] == 3).mean():.3f}")

### Step 4 — compare imputation strategies for Age

**Predict.** After filling 137 holes with one value, will the standard deviation of the ages go up, stay the same or go down? Which age will become the most frequent one?

In [ ]:
from sklearn.impute import SimpleImputer

cols = ['Age', 'Pclass', 'SibSp', 'Parch']
imputers = {'mean': SimpleImputer(strategy='mean'),
            'median': SimpleImputer(strategy='median')}
observed = X_train['Age'].dropna()
rows = {'observed only': [len(observed), observed.mean(), observed.std(), observed.min(),
                          observed.value_counts().iloc[0]]}
for name, imp in imputers.items():
    ages = pd.Series(imp.fit(X_train[cols]).transform(X_train[cols])[:, 0])
    rows[name] = [len(ages), ages.mean(), ages.std(), ages.min(), ages.value_counts().iloc[0]]
print(pd.DataFrame(rows, index=['n', 'mean', 'std', 'min', 'largest pile']).T.round(2))

**Simulation: the shrinkage law, on holes you make.** Take the 575 real ages of the training rows and erase 20%, 50% and 80% of them at random (MCAR). Fill the holes with the mean and compare the standard deviation with the law of section 6, the standard deviation of the kept ages times \(\sqrt{(n_{\text{kept}} - 1)/(n - 1)}\). Does the law hold every time? How far does the filled column drift from the 14.49 of the real ages?

In [ ]:
# your simulation here

### Step 5 — encode Sex, Embarked and Pclass

**Predict.** Three categorical columns with 2, 3 and 3 levels: how many one-hot columns will they become, and how many 1s will each row have?

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['Sex', 'Embarked', 'Pclass']
cat_imp = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
cats_train = cat_imp.fit_transform(X_train[cat_cols])            # learns the modes on train
cats_test = cat_imp.transform(X_test[cat_cols])
print("modes learned on train:", cat_imp.statistics_)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(cats_train)
print(ohe.get_feature_names_out())
print(X_test[cat_cols].head(3).to_numpy().tolist())
print(ohe.transform(cats_test.head(3)))

### Step 6 — scale the numeric columns

**Predict.** In raw units, which numeric column will own the distance between two passengers? And after scaling with the training statistics, will the test columns have mean exactly 0?

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ['Age', 'SibSp', 'Parch', 'Fare']
num_imp = SimpleImputer(strategy='median').set_output(transform='pandas').fit(X_train[num_cols])
num_train, num_test = num_imp.transform(X_train[num_cols]), num_imp.transform(X_test[num_cols])
scaler = StandardScaler().set_output(transform='pandas').fit(num_train)
z_train, z_test = scaler.transform(num_train), scaler.transform(num_test)

share = lambda M: (100 * M.var() / M.var().sum()).round(1)     # share of the squared distance
print(pd.DataFrame({'raw %': share(num_train), 'scaled %': share(z_train)}))
print(pd.DataFrame({'train mean': z_train.mean(), 'train std': z_train.std(ddof=0),
                    'test mean': z_test.mean(), 'test std': z_test.std(ddof=0)}).round(3))

**Simulation: whose neighbors?** Take the first test passenger and find the 5 training passengers closest to them on the four numeric columns, once in raw units (`num_train`, `num_test`) and once scaled (`z_train`, `z_test`). How many passengers appear in both lists? What do the neighbors of each list have in common?

In [ ]:
# your simulation here

**To close the fundamental part.** In one sentence each: (1) why is a k-NN on raw Titanic columns, in practice, a k-NN on fare? (2) Where does the median of step 4 come from, and what do the test passengers' missing ages receive? (3) Why are the scaled test columns of step 6 not exactly mean 0 and standard deviation 1, and why is that correct?

### Your turn (fundamental)

At home:

1. **Another scaler.** Redo step 6 with `RobustScaler` instead of `StandardScaler`. Print its `scale_`: which columns get a spread of exactly 1, and why, for each one? How do the shares of the squared distance change?
2. **Fare's shape.** Compute the skewness of `X_train['Fare']`, of `np.log1p(X_train['Fare'])` and of the logged column after a `StandardScaler`. Which step changes the shape, and which does not?
3. **Pclass as ordinal.** In step 5, leave `Pclass` out of the one-hot and keep it as a single column (1, 2, 3). How many columns does the encoding produce now, and what does this choice assume about the three classes?

## Advanced lab — one object, a black-box model and smarter imputers

### Step 7 — everything in one object: a ColumnTransformer preview

**Predict.** Steps 5 and 6 used four fitted objects. Bundled into one, how many columns will the output have, and will any `NaN` be left in the test rows?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

preprocess = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(handle_unknown='ignore')), cat_cols)])
M_train = preprocess.fit_transform(X_train)       # every statistic is learned here, on train only
M_test = preprocess.transform(X_test)             # ... and only applied here
print(M_train.shape, M_test.shape, "NaN left in test:", int(np.isnan(M_test).sum()))
print(preprocess.get_feature_names_out())
print(M_test[:2].round(2))

### Step 8 — does it matter? A black-box k-NN with and without scaling

**Predict.** The baseline that always answers "did not survive" gets 61.5%. Will a k-NN on unscaled columns be far above it? How much will scaling add?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, RobustScaler

baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"baseline (always 'did not survive')   test accuracy {baseline.score(X_test, y_test):.3f}")
for name, scaler in [('no scaling', 'passthrough'), ('StandardScaler', StandardScaler()),
                     ('MinMaxScaler', MinMaxScaler()), ('RobustScaler', RobustScaler())]:
    prep = ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median'), scaler), num_cols),
        ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                              OneHotEncoder(handle_unknown='ignore')), cat_cols)])
    knn = make_pipeline(prep, KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
    print(f"k-NN (black box), {name:15s}     test accuracy {knn.score(X_test, y_test):.3f}")

**Simulation: shape before scale.** Fare is the most skewed column. Give it a branch of its own in the column transformer: median, then `FunctionTransformer(np.log1p)`, then `StandardScaler`, with the other numeric columns in the usual branch. Does the skewness of the training fares drop? Does the k-NN of step 8 improve?

In [ ]:
# your simulation here

### Step 9 — imputers that use the other columns

**Predict.** An imputer that looks at class and family size: will its imputed ages spread more or less than the median's? Can it produce an impossible age?

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer

others = {'knn': KNNImputer(n_neighbors=5),
          'iterative': IterativeImputer(random_state=0, min_value=0)}
for name, imp in others.items():
    ages = pd.Series(imp.fit(X_train[cols]).transform(X_train[cols])[:, 0])
    rows[name] = [len(ages), ages.mean(), ages.std(), ages.min(), ages.value_counts().iloc[0]]
print(pd.DataFrame(rows, index=['n', 'mean', 'std', 'min', 'largest pile']).T.round(2))

**Simulation: a smarter imputer inside the pipeline.** In the numeric branch of the step-8 pipeline (with `StandardScaler`), replace the median by `KNNImputer(n_neighbors=5)`, once on raw units and once between two scalers (the imputer measures distances, so section 12 says it needs comparable scales). Does the k-NN's test accuracy improve?

In [ ]:
# your simulation here

**To close the lesson.** In one sentence each: (1) what does a `ColumnTransformer` guarantee that four separately fitted objects do not? (2) Why can a log that fixes the shape of the fare still make the k-NN worse? (3) Why does an imputer that keeps more of the spread not necessarily give a better model?

### Your turn (advanced)

At home, with more time:

1. **The missing flag.** Use `SimpleImputer(strategy='median', add_indicator=True)` in the numeric branch. How many columns does the output have now, and what is the name of the new one? Does the k-NN of step 8 change?
2. **Titles.** Extract the title from `Name` (`df['Name'].str.extract(r',\s*([^.]+)\.')`), add it to the features and to `cat_cols`, and encode it with `OneHotEncoder(handle_unknown='ignore', min_frequency=10)`. Then impute `Age` with the **median age of each title**, learned on the training rows (`groupby('Title')['Age'].median()`). What does it give to the boys titled *Master* whose age is missing (3 of them in the training rows), compared with the global median of 28.5?
3. **The leak, in numbers.** Fit `preprocess` on all 891 rows instead of `X_train`. Compare the learned median age and the fare mean and standard deviation (`preprocess.named_transformers_['num']`) with the train-only version. Which statistic moves most, and why is that the one that depends on a few extreme rows?